# Autoencoder Feature Weightages

This notebook trains an autoencoder on selected Excel metrics and outputs normalized feature weightages as JSON.

In [ ]:
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import PowerTransformer, RobustScaler, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split


In [ ]:
excel_path = Path("YOUR_EXCEL_PATH.xlsx")
metrics = ["metric_a", "metric_b", "metric_c"]  # update with desired columns
output_path = Path("weightages.json")
latent_dim = 2
epochs = 200
batch_size = 32
learning_rate = 1e-3
seed = 42
device = "cpu"
validation_split = 0.2
patience = 20
min_delta = 1e-4


In [ ]:
data = pd.read_excel(excel_path)
if metrics:
    missing = [metric for metric in metrics if metric not in data.columns]
    if missing:
        raise ValueError("Metrics not found in Excel data: " + ", ".join(sorted(missing)))
    data = data[list(metrics)]

numeric_data = data.apply(pd.to_numeric, errors="coerce").dropna()
if numeric_data.empty:
    raise ValueError("No numeric rows available after cleaning the data.")
skewness = numeric_data.skew().abs().max()
values = numeric_data.values
power_transformer = None
if skewness > 1.0:
    power_transformer = PowerTransformer(method="yeo-johnson", standardize=False)
    values = power_transformer.fit_transform(values)
    scaler = RobustScaler()
else:
    scaler = StandardScaler()
features = scaler.fit_transform(values).astype(np.float32)


In [ ]:
if latent_dim <= 0:
    raise ValueError("latent_dim must be a positive integer.")
if latent_dim >= features.shape[1]:
    latent_dim = max(1, features.shape[1] // 2)
if latent_dim == 2 and features.shape[1] > 4:
    latent_dim = min(8, max(2, features.shape[1] // 2))
if not 0.0 < validation_split < 1.0:
    raise ValueError("validation_split must be between 0 and 1 (exclusive).")
sample_count = features.shape[0]
if sample_count < 200:
    epochs = max(50, min(epochs, 300))
elif sample_count > 5000:
    epochs = min(epochs, 150)
batch_size = min(batch_size, max(1, min(128, sample_count)))


In [ ]:
class AutoEncoder(nn.Module):
    def __init__(self, input_dim: int, latent_dim: int):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, latent_dim),
            nn.ReLU(),
        )
        self.decoder = nn.Linear(latent_dim, input_dim)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        latent = self.encoder(inputs)
        return self.decoder(latent)


def train_autoencoder(features: np.ndarray) -> tuple[AutoEncoder, float]:
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = AutoEncoder(input_dim=features.shape[1], latent_dim=latent_dim).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_fn = nn.MSELoss()

    dataset = TensorDataset(torch.from_numpy(features).float())
    val_size = max(1, int(len(dataset) * validation_split))
    train_size = max(1, len(dataset) - val_size)
    if train_size + val_size > len(dataset):
        val_size = len(dataset) - train_size
    train_dataset, val_dataset = random_split(
        dataset,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(seed),
    )
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    best_state = None
    best_val = float("inf")
    epochs_without_improve = 0

    for _ in range(epochs):
        model.train()
        for (batch,) in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            reconstruction = model(batch)
            loss = loss_fn(reconstruction, batch)
            loss.backward()
            optimizer.step()

        model.eval()
        val_losses = []
        with torch.no_grad():
            for (batch,) in val_loader:
                batch = batch.to(device)
                reconstruction = model(batch)
                val_losses.append(loss_fn(reconstruction, batch).item())
        val_loss = float(np.mean(val_losses)) if val_losses else float("inf")

        if best_val - val_loss > min_delta:
            best_val = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improve = 0
        else:
            epochs_without_improve += 1
            if epochs_without_improve >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_val


In [ ]:
model, validation_loss = train_autoencoder(features)
validation_loss


In [ ]:
encoder_layer = model.encoder[0]
weights = encoder_layer.weight.detach().cpu().numpy()
importance = np.mean(np.abs(weights), axis=0)
total = float(np.sum(importance))
if math.isclose(total, 0.0):
    normalized = np.zeros_like(importance)
else:
    normalized = importance / total

paired = [(feature, float(weight)) for feature, weight in zip(data.columns, normalized, strict=False)]
paired.sort(key=lambda item: item[1], reverse=True)
weightages = dict(paired)
weightages


In [ ]:
output_path.write_text(json.dumps(weightages, indent=2), encoding="utf-8")
output_path
